In [1]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os, psutil, gc
import time 
import json
import pprint
import signal

from collections import defaultdict
import random

import numpy as np
np.set_printoptions(precision=4)
from scipy.stats import ttest_rel

In [2]:
from sal.config import Config

from datasets import load_dataset

from utils.load_data import load_data_prm800k
from utils import parser, grader2

In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_tokenizer_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_tokenizer_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"

In [4]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)


# ds_completions = load_completions(completions_dir)

# load random_seeds     
# random_seeds = np.loadtxt("random_seeds.txt").astype("int64")
# random_seeds = [int(seed) for seed in random_seeds]

1: 43
2: 90
3: 105
4: 128
5: 134


In [36]:
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

def run_with_timeout(fn_extract_answer, fn_grade, completion, gt_answer, timeout=2):
    # Set the signal handler for SIGALRM
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
    try:
        c_answer = fn_extract_answer(completion, 'math')
        # print(c_answer)
        result = fn_grade(c_answer, gt_answer)
    except TimeoutException:
        print(f"Timeout: {completion}")
        c_answer = None
        result = None
    finally:
        signal.alarm(0)  # Cancel alarm if function returns early
    return c_answer, result

def evaluate_correctness_hf(data_dir, level, n=8, timeout=2):

    # dataset = load_dataset(dataset_name, name=config_name, split=dataset_split, cache_dir=data_dir)
    dataset = load_dataset("json", data_files = data_dir, split='train')
    dataset_by_level = dataset.filter(lambda example: example['level'] == level)

    passn_correctness = np.zeros((len(dataset_by_level)))
    pass1b_correctness = np.zeros((len(dataset_by_level)))
    naive1b_correctness = np.zeros((len(dataset_by_level)))
    weighted1b_correctness = np.zeros((len(dataset_by_level)))
    maj1b_correctness = np.zeros((len(dataset_by_level)))
    
    pass1b_ncomps = np.zeros((len(dataset_by_level)))
    pass1b_lengths = np.zeros((len(dataset_by_level)))

    pass1b_nphases = np.zeros((len(dataset_by_level)))
    pass1b_ndepths = np.zeros((len(dataset_by_level)))
    for q_idx, data in enumerate(dataset_by_level):
        # print(f"q_idx = {q_idx}")
        # passn_completions = data["completions"][:n]
        pass1b_completions = data["completions"]

        # gt_answer = data['answer']
        gt_cot, gt_answer  = parser.parse_ground_truth(data, 'math')
        # print(f"done1")
        naive1b_answer = parser.extract_answer(data[f"pred_naive@{8}"], 'math')
        weighted1b_answer = parser.extract_answer(data[f"pred_weighted@{8}"], 'math')
        maj1b_answer = parser.extract_answer(data[f"pred_maj@{8}"], 'math')
        # print(f"done2")

        # naive1b_correct = grader2.math_equal(naive1b_answer, gt_answer)
        # weighted1b_correct = grader2.math_equal(weighted1b_answer, gt_answer)
        # maj1b_correct = grader2.math_equal(maj1b_answer, gt_answer)
        
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
        try:
            naive1b_correct = grader2.math_equal(naive1b_answer, gt_answer)
            # print(c_answer)
        except TimeoutException:
            print(f"Timeout: {completion}")
            naive1b_correct = False
        finally:
            signal.alarm(0)  # Cancel alarm if function returns early

        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
        try:
            weighted1b_correct = grader2.math_equal(weighted1b_answer, gt_answer)
            # print(c_answer)
        except TimeoutException:
            weighted1b_correct = False
        finally:
            signal.alarm(0)  # Cancel alarm if function returns early

        
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
        try:
            maj1b_correct = grader2.math_equal(maj1b_answer, gt_answer)
            # print(c_answer)
        except TimeoutException:
            maj1b_correct = False
        finally:
            signal.alarm(0)  # Cancel alarm if function returns early

        # print(f"done3")
        passn_correct = False
        pass1b_correct = False
        for cidx, completion in enumerate(pass1b_completions):
            c_answer, is_correct = run_with_timeout(parser.extract_answer, grader2.math_equal, completion, gt_answer)
            if is_correct is True: 
                passn_correct = True
                pass1b_correct = True
                break

        pass1b_total_len = 0
        for cidx, completion in enumerate(pass1b_completions):
            pass1b_total_len += len(completion.split("\n\n"))
            # if len(completion.split("\n\n")) > 35:
            #     print(len(completion.split("\n\n")))
            #     print(f"q_idx = {q_idx}")
        
        passn_correctness[q_idx] = passn_correct
        pass1b_correctness[q_idx] = pass1b_correct
        naive1b_correctness[q_idx] = naive1b_correct
        weighted1b_correctness[q_idx] = weighted1b_correct
        maj1b_correctness[q_idx] = maj1b_correct

        # print(pass1b_total_len)
        # print(len(pass1b_completions))
        pass1b_ncomps[q_idx] = len(pass1b_completions)
        pass1b_lengths[q_idx] = pass1b_total_len/len(pass1b_completions) if len(pass1b_completions) > 0 else 0


    # stop
    # passn_correctness = np.mean(passn_correctness)
    # pass1b_correctness = np.mean(pass1b_correctness)
    # naive1b_correctness = np.mean(naive1b_correctness)
    # weighted1b_correctness = np.mean(weighted1b_correctness)
    # maj1b_correctness = np.mean(maj1b_correctness)
        
    return passn_correctness, pass1b_correctness, naive1b_correctness, weighted1b_correctness, maj1b_correctness, pass1b_ncomps, pass1b_lengths 

def extract_completions_info(data_dir):

    with open(data_dir, 'r', encoding = 'utf-8') as fin:
        for line in fin:
            trial_data = json.loads(line)
            num_questions = len(trial_data["last_phases"])

            pass1b_nphases = np.zeros((num_questions))
            pass1b_ndepths = np.zeros((num_questions))
            for q_idx in range(num_questions):
                if trial_data["last_phases"][q_idx] == 999:
                    # print(q_idx)
                    continue
                pass1b_nphases[q_idx] = trial_data["last_phases"][q_idx]
                pass1b_ndepths[q_idx] = np.mean(trial_data["ndepths_arr"][q_idx])
            
    return pass1b_nphases, pass1b_ndepths
    
# general params
config = Config()

level = 4
num_trials = 5

config_name = "mcts--level-4--e21--n-4--d-20--nb-4--cpuct-2"
result_dir = f"results/mcts--level-{level}/{config_name}"
print(f"config_name = {config_name}")

all_passn_correctness = []
all_pass1b_correctness = []
all_naive1b_correctness = []
all_weighted1b_correctness = []
all_maj1b_correctness = []

all_pass1b_ncomps = []
all_pass1b_lengths = []
for trial_idx in range(num_trials):
    passn_correctness, pass1b_correctness, naive1b_correctness, weighted1b_correctness, maj1b_correctness, pass1b_ncomps, pass1b_lengths = \
        evaluate_correctness_hf(f"{result_dir}/{config_name}--trial-{trial_idx}.jsonl", 
                                level, config.n)

    all_passn_correctness.append(passn_correctness)
    all_pass1b_correctness.append(pass1b_correctness)
    all_naive1b_correctness.append(naive1b_correctness)
    all_weighted1b_correctness.append(weighted1b_correctness)
    all_maj1b_correctness.append(maj1b_correctness)

    all_pass1b_ncomps.append(pass1b_ncomps)
    all_pass1b_lengths.append(pass1b_lengths)

all_passn_correctness = np.concatenate(all_passn_correctness)
all_pass1b_correctness = np.concatenate(all_pass1b_correctness)
all_naive1b_correctness = np.concatenate(all_naive1b_correctness)
all_weighted1b_correctness = np.concatenate(all_weighted1b_correctness)
all_maj1b_correctness = np.concatenate(all_maj1b_correctness)

all_pass1b_ncomps = np.concatenate(all_pass1b_ncomps)
all_pass1b_lengths = np.concatenate(all_pass1b_lengths)

print(all_pass1b_correctness.shape)
np.savetxt(f"{result_dir}/passn_{config_name}.txt", all_passn_correctness)
np.savetxt(f"{result_dir}/pass1b_{config_name}.txt", all_pass1b_correctness)
np.savetxt(f"{result_dir}/naive1b_{config_name}.txt", all_naive1b_correctness)
np.savetxt(f"{result_dir}/weighted1b_{config_name}.txt", all_weighted1b_correctness)
np.savetxt(f"{result_dir}/maj1b_{config_name}.txt", all_maj1b_correctness)

passn_correctness_mean = np.mean(all_passn_correctness)
pass1b_correctness_mean = np.mean(all_pass1b_correctness)
naive1b_correctness_mean = np.mean(all_naive1b_correctness)
weighted1b_correctness_mean = np.mean(all_weighted1b_correctness)
maj1b_correctness_mean = np.mean(all_maj1b_correctness)

num_questions = len(data_by_levels[level])
passn_correctness_std = np.std(all_passn_correctness, ddof=1)/np.sqrt(num_trials*num_questions) # 128 is number of prompts for level 4 
pass1b_correctness_std = np.std(all_pass1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
naive1b_correctness_std = np.std(all_naive1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
weighted1b_correctness_std = np.std(all_weighted1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
maj1b_correctness_std = np.std(all_maj1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)

pass1b_ncomps_mean = np.mean(all_pass1b_ncomps)
pass1b_lengths_mean = np.mean(all_pass1b_lengths)
pass1b_ncomps_std = np.std(all_pass1b_ncomps, ddof=1)/np.sqrt(num_trials*num_questions)
pass1b_lengths_std = np.std(all_pass1b_lengths, ddof=1)/np.sqrt(num_trials*num_questions)
                
all_pass1b_nphases = []
all_pass1b_ndepths = []
for trial_idx in range(0,num_trials):
    data_dir = f"{result_dir}/generate_{config_name}--trial-{trial_idx}.jsonl"
    pass1b_nphases, pass1b_ndepths = extract_completions_info(data_dir)
    all_pass1b_nphases.append(pass1b_nphases)
    all_pass1b_ndepths.append(pass1b_ndepths)

all_pass1b_nphases = np.concatenate(all_pass1b_nphases)
all_pass1b_ndepths = np.concatenate(all_pass1b_ndepths)

pass1b_nphases_mean = np.mean(all_pass1b_nphases)
pass1b_ndepths_mean = np.mean(all_pass1b_ndepths)
pass1b_nphases_std = np.std(all_pass1b_nphases, ddof=1)/np.sqrt(num_trials*num_questions)
pass1b_ndepths_std = np.std(all_pass1b_ndepths, ddof=1)/np.sqrt(num_trials*num_questions)

# print(f"pass1b_nphases: {pass1b_nphases_mean:0.1f} (\u00B1{pass1b_nphases_std:0.1f})")
# print(f"pass1b_ndepths: {pass1b_ndepths_mean:0.1f} (\u00B1{pass1b_ndepths_std:0.1f})")
print(
    f"{pass1b_correctness_mean:0.4f} (\u00B1{pass1b_correctness_std:0.4f}), "
    f"{naive1b_correctness_mean:0.4f} (\u00B1{naive1b_correctness_std:0.4f}),"
    f"{weighted1b_correctness_mean:0.4f} (\u00B1{weighted1b_correctness_std:0.4f}), "
    f"{maj1b_correctness_mean:0.4f} (\u00B1{maj1b_correctness_std:0.4f}), "
    f"{pass1b_ncomps_mean:0.1f}    (\u00B1{pass1b_ncomps_std:0.1f}), "
    f"{pass1b_lengths_mean:0.1f}    (\u00B1{pass1b_lengths_std:0.1f}), "
    f"{pass1b_nphases_mean:0.1f}    (\u00B1{pass1b_nphases_std:0.1f}), "
    f"{pass1b_ndepths_mean:0.1f}    (\u00B1{pass1b_ndepths_std:0.1f})"
)

config_name = mcts--level-4--e21--n-4--d-20--nb-4--cpuct-2


Filter:   0%|          | 0/128 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128 [00:00<?, ? examples/s]

(640,)
0.6203 (±0.0192), 0.4297 (±0.0196),0.4578 (±0.0197), 0.4422 (±0.0196), 11.6    (±0.3), 8.7    (±0.2), 9.2    (±0.2), 9.7    (±0.2)
